In [1]:
!pip install -q --upgrade trl transformers peft bitsandbytes accelerate wandb huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 15.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 69.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 95.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2

In [2]:

import os
import torch
import wandb
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

os.environ["HF_TOKEN"] = "your_api_key_here"
os.environ["WANDB_API_KEY"] = "Your_api_key_here"

login(token=os.environ["HF_TOKEN"])
wandb.login(key=os.environ["WANDB_API_KEY"])

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tesla369 (tesla369-pdepth) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU: Tesla T4
Memory: 15.6 GB


In [3]:

dataset = load_dataset("TeslaInch/SCD-Instruction-Tuning")

print(dataset)
print(f"\nTrain: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"\nSample:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/461 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/337k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/41.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3487 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/387 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 3487
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 387
    })
})

Train: 3487
Validation: 387

Sample:
{'instruction': 'Answer the following question about sickle cell disease.', 'input': 'What is the prevalence of cardiac enlargement in individuals with sickle cell disease?', 'output': 'Between 22 percent and 76 percent.'}


In [8]:
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Phi-3.5 Mini...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

print(f"Loaded — GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading Phi-3.5 Mini...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Loaded — GPU: 3.33 GB


In [9]:

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [10]:

wandb.finish()
wandb.init(
    project="scd-medical-llm",
    name="phi35-mini-scd-v7",
    config={
        "model": "Phi-3.5-mini-instruct",
        "dataset": "TeslaInch/SCD-Instruction-Tuning",
        "version": "v7",
        "train_examples": len(train_formatted),
        "val_examples": len(val_formatted),
        "lora_rank": 16,
        "lora_alpha": 32,
        "epochs": 3,
    }
)

sft_config = SFTConfig(
    output_dir="./scd-phi35-adapter-v7",
    num_train_epochs=3,               # reduced from 5 — less forgetting
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    learning_rate=1e-4,               # reduced from 2e-4 — more conservative
    weight_decay=0.05,                # increased from 0.01 — less forgetting
    warmup_steps=20,
    lr_scheduler_type="cosine",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
)

print("Starting training run 7...")
print(f"Train examples: {len(train_formatted)}")
print(f"Val examples: {len(val_formatted)}")
print(f"Epochs: 3")
print(f"Steps per epoch: {len(train_formatted) // sft_config.per_device_train_batch_size}")

trainer.train()

Starting training run 7...
Train examples: 3487
Val examples: 387
Epochs: 3
Steps per epoch: 1743


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,2.796180,2.523443,1.336077,47036.000000,0.714826
100,2.207046,2.252779,1.110228,93765.000000,0.740694
150,2.173846,2.222239,1.076022,141358.000000,0.743810
200,2.162709,2.204180,1.086977,188723.000000,0.746097
250,2.218764,2.193210,1.103832,235806.000000,0.745615
300,2.063710,2.186341,1.063581,282994.000000,0.746721
350,2.181444,2.176693,1.070888,330282.000000,0.747420
400,2.180961,2.174070,1.080549,377296.000000,0.746173
450,2.107643,2.168043,1.068539,424326.000000,0.747402
500,1.998622,2.168982,1.056789,471302.000000,0.747879


TrainOutput(global_step=654, training_loss=2.346081496014143, metrics={'train_runtime': 10155.7377, 'train_samples_per_second': 1.03, 'train_steps_per_second': 0.064, 'total_flos': 1.6835014794203136e+16, 'train_loss': 2.346081496014143, 'epoch': 3.0})

In [11]:
trainer.model.push_to_hub(
    "TeslaInch/scd-phi35-adapter-v7",
    token="your_token_here"
)
tokenizer.push_to_hub(
    "TeslaInch/scd-phi35-adapter-v7",
    token="your_token_here"
)
wandb.finish()
print("V7 adapter pushed")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

eval/entropy,█▂▁▂▂▁▁▂▁▁▁▁▁▁
eval/loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁
eval/mean_token_accuracy,▁▆▇███████████
eval/num_tokens,▁▂▂▃▃▄▄▅▆▆▇▇██
eval/runtime,▅▇▆▆▅▁▄▅▁▄▅█▅▆
eval/samples_per_second,▄▃▃▄▄█▅▅█▅▄▁▄▄
eval/steps_per_second,▅▃▃▃▅█▆▅█▅▅▁▃▃
train/entropy,▃▆█▃▂▂▃▂▂▂▂▂▂▂▂▁▂▂▂▁▂▂▂▂▂▂▂▂▁▂▁▁▂▁▂▂▂▂▂▂
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
+5,...


V7 adapter pushed
